# Sparse Coding de Espectrogramas STFT — ESC-50
## Aprendizaje de Átomos Espectrales para Sonidos Ambientales

In [ ]:
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import os
import glob
import pandas as pd
import IPython.display as ipd

from sklearn.decomposition import MiniBatchDictionaryLearning, SparseCoder
from scipy.signal import find_peaks

np.random.seed(42)

# Parámetros globales
SR           = 22050
N_FFT        = 1024
HOP_LENGTH   = 256
MAX_DURATION = 5.0
N_COMPONENTS = 128
BATCH_SIZE   = 2048
ALPHA        = 1
MAX_ITER     = 50
THRESHOLD    = 1e-3

DATASET_DIR  = r"ESC-50-master\ESC-50-master\audio"

print("Librerías cargadas.")

## Paso 1 — Carga del dataset ESC-50

In [ ]:
def load_esc50(dataset_dir, sr=SR, max_dur=MAX_DURATION):
    paths = sorted(glob.glob(os.path.join(dataset_dir, "*.wav")))
    print(f"Archivos encontrados: {len(paths)}")

    signals  = []
    metadata = []

    for i, p in enumerate(paths):
        y, _ = librosa.load(p, sr=sr, mono=True)
        y = y[:int(max_dur * sr)]
        signals.append(y)

        fname = os.path.basename(p)
        parts = fname.replace('.wav', '').split('-')
        metadata.append({
            'path': p,
            'filename': fname,
            'fold':   int(parts[0]),
            'target': int(parts[3]) if len(parts) >= 4 else -1
        })

        if (i + 1) % 200 == 0:
            print(f"  {i+1}/{len(paths)} cargados...")

    df = pd.DataFrame(metadata)
    print(f"\nTotal: {len(signals)} señales — {df['target'].nunique()} clases")
    return signals, df


signals, df_meta = load_esc50(DATASET_DIR)
print(df_meta.head())

## Paso 2 — Visualización de muestras del dataset

In [ ]:
N_SHOW = 6

fig, axes = plt.subplots(N_SHOW, 1, figsize=(14, N_SHOW * 2.5))
fig.suptitle("Muestras del Dataset ESC-50 — Forma de onda", fontsize=13, fontweight='bold')

for i, (y, ax) in enumerate(zip(signals[:N_SHOW], axes)):
    librosa.display.waveshow(y, sr=SR, ax=ax, color='steelblue', alpha=0.8)
    ax.set_title(f"#{i} — {df_meta.iloc[i]['filename']}", fontsize=9)
    ax.set_xlabel("Tiempo (s)", fontsize=8)
    ax.set_ylabel("Amplitud", fontsize=8)
    ax.tick_params(labelsize=7)

plt.tight_layout()
plt.show()

print("\n— Reproductores de audio —\n")
for i, y in enumerate(signals[:N_SHOW]):
    print(f"#{i} — {df_meta.iloc[i]['filename']}")
    display(ipd.Audio(y, rate=SR))

## Paso 3 — STFT, magnitud y log-compresión

Pipeline: señal → STFT → |·| → log1p → X  
Sin StandardScaler: log1p estabiliza el rango dinámico y mantiene los valores no-negativos,
lo que permite invertir directamente con expm1 para la reconstrucción.

In [ ]:
def signal_to_frames(y, n_fft=N_FFT, hop_length=HOP_LENGTH):
    """señal → (T, F) en espacio log1p."""
    S = librosa.stft(y, n_fft=n_fft, hop_length=hop_length, window='hann')
    S = np.abs(S)
    S = np.log1p(S)
    return S.T.astype(np.float32)  # (T, F)


BATCH_FILES       = 200
MAX_FRAMES_TOTAL  = 200_000

all_frames        = []
first_spectrogram = None
total_frames      = 0

for batch_start in range(0, len(signals), BATCH_FILES):
    batch        = signals[batch_start:batch_start + BATCH_FILES]
    batch_frames = []

    for y in batch:
        frames = signal_to_frames(y)
        batch_frames.append(frames)
        if first_spectrogram is None:
            first_spectrogram = frames

    chunk = np.vstack(batch_frames)
    all_frames.append(chunk)
    total_frames += chunk.shape[0]
    print(f"  Batch {batch_start//BATCH_FILES + 1} — frames acumulados: {total_frames}")

    if total_frames >= MAX_FRAMES_TOTAL:
        print("Cap de frames alcanzado.")
        break

X = np.vstack(all_frames)
if X.shape[0] > MAX_FRAMES_TOTAL:
    idx = np.random.choice(X.shape[0], MAX_FRAMES_TOTAL, replace=False)
    idx.sort()
    X = X[idx]

F = X.shape[1]
print(f"\nDataset: {X.shape} — memoria: {X.nbytes/1e6:.1f} MB")

## Paso 4 — Aprendizaje del diccionario

In [ ]:
MAX_TRAIN_FRAMES = 50_000
if X.shape[0] > MAX_TRAIN_FRAMES:
    idx     = np.random.choice(X.shape[0], MAX_TRAIN_FRAMES, replace=False)
    X_train = X[idx]
else:
    X_train = X

dict_learner = MiniBatchDictionaryLearning(
    n_components=N_COMPONENTS,
    alpha=ALPHA,
    batch_size=BATCH_SIZE,
    max_iter=MAX_ITER,
    transform_algorithm='lasso_lars',
    fit_algorithm='cd',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print(f"Entrenando con {X_train.shape[0]} frames...")
dict_learner.fit(X_train)

D = dict_learner.components_  # (N_COMPONENTS, F)
print(f"\nDiccionario D: {D.shape}")

## Paso 5 — Coeficientes sparse y sparsity

In [ ]:
MAX_TRANSFORM = 20_000
X_sub = X[:MAX_TRANSFORM]

print(f"Transformando {X_sub.shape[0]} frames...")
A = dict_learner.transform(X_sub)

density          = np.mean(np.abs(A) > THRESHOLD)
active_per_frame = np.sum(np.abs(A) > THRESHOLD, axis=1)

print(f"Densidad de activación : {density:.4f} ({density*100:.2f}%)")
print(f"Átomos activos/frame   — media: {active_per_frame.mean():.1f}, mediana: {np.median(active_per_frame):.1f}, máx: {active_per_frame.max()}")
if density < 0.1:
    print("✓ Buena sparsity (< 10%)")
else:
    print("⚠ Sparsity alta — considera aumentar alpha")

## Paso 6 — Funciones de reconstrucción de audio

Pipeline de inversión estándar:  
frames log1p → expm1 → magnitud lineal → Griffin-Lim (256 iter) → audio

In [ ]:
def frames_to_audio(frames_log, n_iter=256, hop_length=HOP_LENGTH, n_fft=N_FFT):
    """
    frames_log: (T, F) en espacio log1p
    Devuelve señal de audio normalizada.
    """
    S_mag = np.expm1(np.maximum(frames_log, 0)).T  # (F, T) magnitud lineal
    y = librosa.griffinlim(
        S_mag,
        n_iter=n_iter,
        hop_length=hop_length,
        win_length=n_fft,
        window='hann'
    )
    y = y / (np.max(np.abs(y)) + 1e-8)
    return y


def atom_to_audio(atom, duration=2.0, sr=SR, n_iter=256, hop_length=HOP_LENGTH, n_fft=N_FFT):
    """
    Sintetiza un átomo (perfil espectral log1p) como audio.
    Repite el perfil N frames y aplica Griffin-Lim.
    """
    n_frames = int(duration * sr / hop_length)
    frames   = np.tile(atom, (n_frames, 1))  # (T, F)
    return frames_to_audio(frames, n_iter=n_iter, hop_length=hop_length, n_fft=n_fft)


print("Funciones de reconstrucción definidas.")

## Paso 7 — Visualización y reproducción de 20 átomos aleatorios

In [ ]:
freqs    = librosa.fft_frequencies(sr=SR, n_fft=N_FFT)
rng      = np.random.default_rng(0)
rand_idx = rng.choice(N_COMPONENTS, size=20, replace=False)

fig, axes = plt.subplots(4, 5, figsize=(18, 10))
fig.suptitle("20 Átomos Espectrales Aleatorios — ESC-50", fontsize=14, fontweight='bold')

for ax, k in zip(axes.flat, rand_idx):
    ax.plot(freqs / 1000, D[k], lw=0.8, color='steelblue')
    ax.set_title(f"Átomo #{k}", fontsize=8)
    ax.set_xlabel("kHz", fontsize=7)
    ax.tick_params(labelsize=6)
    ax.axhline(0, color='gray', lw=0.5, ls='--')

plt.tight_layout()
plt.show()

print("\n— Reproducción de átomos aleatorios —\n")
for k in rand_idx:
    y_atom = atom_to_audio(D[k])
    if np.any(np.isnan(y_atom)) or np.max(np.abs(y_atom)) < 1e-8:
        print(f"Átomo #{k} — silencio")
        display(ipd.Audio(np.zeros(SR), rate=SR))
    else:
        print(f"Átomo #{k}")
        display(ipd.Audio(y_atom, rate=SR))

## Paso 8 — Top 8 átomos más utilizados

In [ ]:
usage      = np.sum(np.abs(A) > THRESHOLD, axis=0)
idx_sorted = np.argsort(usage)[::-1]
top_atoms  = idx_sorted[:8]

print("Top 8 átomos más utilizados:")
for rank, k in enumerate(top_atoms):
    print(f"  Rank {rank+1}: átomo #{k}  →  {usage[k]} frames")

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
fig.suptitle("Top 8 Átomos Más Utilizados — ESC-50", fontsize=14, fontweight='bold')

for ax, k in zip(axes.flat, top_atoms):
    ax.plot(freqs / 1000, D[k], lw=0.8, color='darkorange')
    ax.set_title(f"#{k} (uso={usage[k]})", fontsize=8)
    ax.set_xlabel("kHz", fontsize=7)
    ax.tick_params(labelsize=6)
    ax.axhline(0, color='gray', lw=0.5, ls='--')

plt.tight_layout()
plt.show()

print("\n— Reproducción de átomos más usados —\n")
for k in top_atoms:
    y_atom = atom_to_audio(D[k])
    if np.any(np.isnan(y_atom)) or np.max(np.abs(y_atom)) < 1e-8:
        print(f"Átomo #{k} (uso={usage[k]}) — silencio")
        display(ipd.Audio(np.zeros(SR), rate=SR))
    else:
        print(f"Átomo #{k} (uso={usage[k]})")
        display(ipd.Audio(y_atom, rate=SR))

## Paso 9 — Clasificación de átomos por tipo espectral

4 tipos relevantes para sonidos ambientales:
- **Broadband**: energía repartida en todo el espectro → lluvia, viento, ruido
- **Lowpass**: energía concentrada en graves → trueno, motor
- **Periódico**: picos con espaciado regular → pájaros, sirenas
- **Transiente**: pocos picos muy prominentes → golpes, pasos

In [ ]:
def classify_atom(atom, sr=SR):
    mag      = np.abs(atom)
    mag_norm = mag / (mag.max() + 1e-8)
    cutoff   = int(2000 / (sr / 2) * len(mag))

    e_low  = mag_norm[:cutoff].mean()
    e_high = mag_norm[cutoff:].mean()

    broadband_score = 1 - abs(e_low - e_high)
    lowpass_score   = e_low - e_high

    peaks, _ = find_peaks(mag_norm, height=0.2, distance=5)
    if len(peaks) >= 3:
        diffs          = np.diff(peaks)
        cv             = diffs.std() / (diffs.mean() + 1e-8)
        periodic_score = max(0, 1 - cv)
    else:
        periodic_score = 0.0

    peaks_high, _ = find_peaks(mag_norm, height=0.5)
    transient_score = min(1.0, len(peaks_high) / 5) if 0 < len(peaks_high) <= 5 else 0.0

    scores    = {'broadband': broadband_score, 'lowpass': lowpass_score,
                 'periodico': periodic_score,  'transiente': transient_score}
    categoria = max(scores, key=scores.get)
    return categoria, scores


atom_categories = {}
category_counts = {'broadband': [], 'lowpass': [], 'periodico': [], 'transiente': []}

for k in range(N_COMPONENTS):
    cat, scores = classify_atom(D[k])
    atom_categories[k] = cat
    category_counts[cat].append(k)

print("Distribución de categorías:")
for cat, atoms in category_counts.items():
    print(f"  {cat:12s}: {len(atoms)} átomos")

fig, ax = plt.subplots(figsize=(7, 4))
cats   = list(category_counts.keys())
counts = [len(category_counts[c]) for c in cats]
colors = ['steelblue', 'darkorange', 'mediumseagreen', 'tomato']
ax.bar(cats, counts, color=colors, edgecolor='white')
ax.set_title("Distribución de Tipos de Átomos — ESC-50", fontsize=12, fontweight='bold')
ax.set_ylabel("Número de átomos")
for i, v in enumerate(counts):
    ax.text(i, v + 0.3, str(v), ha='center', fontsize=10)
plt.tight_layout()
plt.show()

## Paso 10 — Visualización y reproducción por categoría

In [ ]:
cat_colors = {
    'broadband':  'steelblue',
    'lowpass':    'darkorange',
    'periodico':  'mediumseagreen',
    'transiente': 'tomato'
}

N_PER_CAT = 4

for cat, color in cat_colors.items():
    atoms_cat = category_counts[cat][:N_PER_CAT]
    if not atoms_cat:
        print(f"No hay átomos de tipo '{cat}'")
        continue

    fig, axes = plt.subplots(1, len(atoms_cat), figsize=(4 * len(atoms_cat), 3))
    if len(atoms_cat) == 1:
        axes = [axes]
    fig.suptitle(f"Átomos tipo '{cat.upper()}'", fontsize=12, fontweight='bold')

    for ax, k in zip(axes, atoms_cat):
        ax.plot(freqs / 1000, D[k], lw=0.9, color=color)
        ax.set_title(f"#{k}", fontsize=9)
        ax.set_xlabel("kHz", fontsize=7)
        ax.axhline(0, color='gray', lw=0.4, ls='--')
        ax.tick_params(labelsize=6)

    plt.tight_layout()
    plt.show()

    print(f"\n— Reproducción: {cat} —\n")
    for k in atoms_cat:
        y_atom = atom_to_audio(D[k])
        if np.any(np.isnan(y_atom)) or np.max(np.abs(y_atom)) < 1e-8:
            print(f"Átomo #{k} — silencio")
            display(ipd.Audio(np.zeros(SR), rate=SR))
        else:
            print(f"Átomo #{k}")
            display(ipd.Audio(y_atom, rate=SR))

## Paso 11 — Reconstrucción de 4 audios con SparseCoder

In [ ]:
N_RECONSTRUCT = 4

coder = SparseCoder(
    dictionary=D,
    transform_algorithm='lasso_lars',
    transform_alpha=ALPHA
)

for IDX in range(N_RECONSTRUCT):
    y_original  = signals[IDX]
    fname       = df_meta.iloc[IDX]['filename']

    frames_orig = signal_to_frames(y_original)  # (T, F) log1p
    A_rec       = coder.transform(frames_orig)  # (T, N_COMPONENTS)
    frames_rec  = A_rec @ D                     # (T, F) log1p reconstruido

    error_rel = np.linalg.norm(frames_orig - frames_rec) / (np.linalg.norm(frames_orig) + 1e-8)

    print(f"\n{'='*55}")
    print(f"Audio #{IDX} — {fname}")
    print(f"Error de reconstrucción: {error_rel*100:.2f}%")
    print(f"{'='*55}")

    fig, axes = plt.subplots(1, 2, figsize=(16, 4))
    vmin, vmax = frames_orig.min(), frames_orig.max()

    axes[0].imshow(frames_orig.T, aspect='auto', origin='lower', cmap='magma', vmin=vmin, vmax=vmax)
    axes[0].set_title(f"Original — {fname}", fontsize=10)
    axes[0].set_xlabel("Frame")
    axes[0].set_ylabel("Bin de frecuencia")

    axes[1].imshow(frames_rec.T, aspect='auto', origin='lower', cmap='magma', vmin=vmin, vmax=vmax)
    axes[1].set_title(f"Reconstruido (error={error_rel*100:.1f}%)", fontsize=10)
    axes[1].set_xlabel("Frame")
    axes[1].set_ylabel("Bin de frecuencia")

    plt.tight_layout()
    plt.show()

    print(f"Audio original #{IDX}:")
    display(ipd.Audio(y_original, rate=SR))

    # expm1 → magnitud lineal → Griffin-Lim 256 iter
    y_rec = frames_to_audio(frames_rec, n_iter=256)

    print(f"Audio reconstruido #{IDX}:")
    display(ipd.Audio(y_rec, rate=SR))

## Paso 12 — Análisis de activaciones por clase ESC-50

In [ ]:
N_CLASSES      = df_meta['target'].nunique()
class_profiles = np.zeros((N_CLASSES, N_COMPONENTS))

MAX_PER_CLASS    = 10
class_signal_map = {}
for _, row in df_meta.iterrows():
    t = row['target']
    if t not in class_signal_map:
        class_signal_map[t] = []
    if len(class_signal_map[t]) < MAX_PER_CLASS:
        class_signal_map[t].append(int(row.name))

print("Calculando perfiles por clase...")
for cls, idxs in class_signal_map.items():
    cls_frames = np.vstack([signal_to_frames(signals[i]) for i in idxs])
    A_cls      = coder.transform(cls_frames)
    class_profiles[cls] = np.mean(np.abs(A_cls) > THRESHOLD, axis=0)

top30 = np.argsort(usage)[::-1][:30]

fig, ax = plt.subplots(figsize=(18, 10))
im = ax.imshow(class_profiles[:, top30], aspect='auto', cmap='YlOrRd')
ax.set_title("Activación media por clase ESC-50 — Top 30 átomos", fontsize=13, fontweight='bold')
ax.set_xlabel("Átomo (top 30 por uso)", fontsize=10)
ax.set_ylabel("Clase ESC-50", fontsize=10)
ax.set_xticks(range(30))
ax.set_xticklabels([f"#{top30[i]}" for i in range(30)], rotation=90, fontsize=7)
plt.colorbar(im, ax=ax, label="Fracción de frames activos")
plt.tight_layout()
plt.show()

## Paso 13 — Tradeoff sparsity vs. error de reconstrucción

In [ ]:
alphas  = [0.1, 0.5, 1.0, 2.0, 5.0]
results = []
X_exp   = X[:5000]

for a in alphas:
    dl = MiniBatchDictionaryLearning(
        n_components=64, alpha=a, batch_size=512, max_iter=30,
        transform_algorithm='lasso_lars', fit_algorithm='cd',
        n_jobs=-1, random_state=42, verbose=0
    )
    dl.fit(X_exp)
    A_exp     = dl.transform(X_exp)
    D_exp     = dl.components_
    X_rec_exp = A_exp @ D_exp

    dens = np.mean(np.abs(A_exp) > THRESHOLD)
    err  = np.linalg.norm(X_exp - X_rec_exp) / (np.linalg.norm(X_exp) + 1e-8)
    results.append((a, dens, err))
    print(f"alpha={a:.1f}  →  densidad={dens:.4f}  error={err:.4f}")

alphas_v, densities, errors = zip(*results)

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()
ax1.plot(alphas_v, densities, 'o-', color='royalblue', label='Densidad')
ax2.plot(alphas_v, errors,    's--', color='tomato',   label='Error relativo')
ax1.set_xlabel("alpha", fontsize=11)
ax1.set_ylabel("Densidad de activación", color='royalblue', fontsize=10)
ax2.set_ylabel("Error de reconstrucción relativo", color='tomato', fontsize=10)
ax1.set_xscale('log')
ax1.set_title("Tradeoff Sparsity vs. Error — ESC-50", fontsize=12, fontweight='bold')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

## Resumen final

In [ ]:
print("=" * 55)
print("RESUMEN — ESC-50 SPARSE CODING")
print("=" * 55)
print(f"Señales cargadas          : {len(signals)}")
print(f"Clases ESC-50             : {df_meta['target'].nunique()}")
print(f"Frames totales            : {X.shape[0]}")
print(f"Bins de frecuencia        : {F}")
print(f"Átomos del diccionario    : {N_COMPONENTS}")
print(f"Alpha                     : {ALPHA}")
print(f"Densidad de activación    : {density:.4f} ({density*100:.2f}%)")
print("")
print("Distribución de tipos de átomos:")
for cat, atoms in category_counts.items():
    print(f"  {cat:12s}: {len(atoms):3d} átomos")
print("=" * 55)